# Laboratório 1 — Classificação multirrótulo de filmes

### Dataframe

In [ ]:
dados = {
    'nome_filme': ['Galáxia em Chamas', 'Rir pra Não Chorar', 'O Resgate 2', 'Amor e Aliens', 'Comédia no Espaço'],
    'explosões': [50, 0, 80, 5, 10],
    'piadas': [5, 40, 2, 10, 35],
    'orçamento_milhoes': [200, 10, 150, 50, 80],
    'generos': [['Ação', 'Ficção'], ['Comédia'], ['Ação'], ['Ficção', 'Romance'], ['Comédia', 'Ficção']]
}

In [ ]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, hamming_loss

# Criar o DataFrame
df = pd.DataFrame(dados)
display(df)

### Binarizador e Matriz

In [ ]:
mlb = MultiLabelBinarizer()

# Transformar a coluna generos em matriz de rótulos
y_binarizado = mlb.fit_transform(df['generos'])

print("Categorias Encontradas:",mlb.classes_)
print("\nMatriz de Rótulos:")
print(y_binarizado)

# Transformar em DataFrame
df_rotulos = pd.DataFrame(
    y_binarizado,
    columns=mlb.classes_
)

# Exibir
display(df_rotulos)

### Classificador

In [ ]:
X = df[['piadas','explosões','orçamento_milhoes']]
y = y_binarizado

floresta = RandomForestClassifier(random_state=42)
floresta.fit(X, y)


### Inclusão de dados

In [ ]:
filmes_novos = [[60,2,180],[2,50,15]]

### Previsões

In [ ]:
previsao_binaria = floresta.predict(filmes_novos)
previsao_texto = mlb.inverse_transform(previsao_binaria)
print("Previsões:")
print(f"Previsão para o filme A: {previsao_texto[0]}")
print(f"Previsão para o filme B: {previsao_texto[1]}")

### Métricas

In [ ]:
y_verdadeiro = [[1,0,1,0]]
y_previsto = [[1,0,0,0]]

# Exact Match
acuracia_exata = accuracy_score(y_verdadeiro, y_previsto)
print(f"Acurácia Exata: {acuracia_exata*100:.0f}%")

# Hamming Loss
perda_hamming = hamming_loss(y_verdadeiro, y_previsto)
print(f"Perda Hamming: {perda_hamming*100:.0f}%")

# Laboratório 2 — Classificação hierárquica de produtos

### Dataframe

In [ ]:
# Criando dados fictícios com dois níveis de categoria
dados2 = {
    'produto': ['Fit LX', 'Corolla', 'Moto G86', 'Galaxy A36', 'MacBook Air'],
    'peso_kg': [1050, 1300, 0.2, 0.19, 1.2],
    'preco_rs': [45000, 120000, 1500, 1400, 8000],
    'bateria_mah': [0, 0, 5000, 4500, 8000],
    'categoria_pai': ['Veículo', 'Veículo', 'Eletrônico', 'Eletrônico', 'Eletrônico'],
    'categoria_filho': ['Hatch', 'Sedan', 'Smartphone', 'Smartphone', 'Notebook']
}

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

# Criar o DataFrame
df = pd.DataFrame(dados2)
display(df)

### Modelos e Treinamentos

In [ ]:
X = df[['peso_kg', 'preco_rs', 'bateria_mah']]
y_pai = df['categoria_pai']

modelo_gerente = DecisionTreeClassifier(max_depth=2, random_state=42)
modelo_gerente.fit(X, y_pai)

In [ ]:
modelo_especialista_veiculo = DecisionTreeClassifier(random_state=42)
df_veiculos = df[df['categoria_pai'] == 'Veículo']
modelo_especialista_veiculo.fit(df_veiculos[['peso_kg', 'preco_rs', 'bateria_mah']], df_veiculos['categoria_filho'])

In [ ]:
modelo_especialista_eletronico = DecisionTreeClassifier(random_state=42)
df_eletronicos = df[df['categoria_pai'] == 'Eletrônico']
modelo_especialista_eletronico.fit(df_eletronicos[['peso_kg', 'preco_rs', 'bateria_mah']], df_eletronicos['categoria_filho'])

### Inclusão de novos produtos

In [ ]:
novo_produto = [[0.18,1300,4000]]

previsao_gerente = modelo_gerente.predict(novo_produto)[0]
print(f"Previsão do Gerente: {previsao_gerente}")

if previsao_gerente == 'Veículo':
    previsao_especialista = modelo_especialista_veiculo.predict(novo_produto)[0]
elif previsao_gerente == 'Eletrônico':
    previsao_especialista = modelo_especialista_eletronico.predict(novo_produto)[0]
else:
    previsao_especialista = 'Categoria não reconhecida'

print(f"Previsão do Especialista: {previsao_especialista}\n\n\n\n\n\n\n")
